# Kaggle GPU 训练内核(自动生成,勿手改)
由 tools/push_training.py 渲染生成。改动请改模板或 agent 代码。

In [ ]:
# ========== EXPERIMENT CONFIGURATION ==========
EXP_NAME = "__EXP_NAME__"
CHILD_EXP_NAME = "__CHILD_EXP_NAME__"
COMPETITION = "__COMPETITION__"
FOLDS_TO_TRAIN = "0,1,2,3,4"
USE_WANDB = False
CREATE_OOF = True
# ==============================================

print(f"Running {EXP_NAME}/{CHILD_EXP_NAME} on {COMPETITION}")

In [ ]:
%cd /kaggle/working
!pip install -q pyyaml pandas scikit-learn pillow pydicom

In [ ]:
# 数据准备:比赛未挂载时下载小文件 CSV 到 /kaggle/working/comp_data,并改写 config 的 data_dir
import pathlib, subprocess, yaml

comp_root = pathlib.Path(f"/kaggle/input/competitions/{COMPETITION}")
if (comp_root / "train.csv").exists():
    print("competition data mounted:", comp_root)
else:
    dst = pathlib.Path("/kaggle/working/comp_data")
    dst.mkdir(parents=True, exist_ok=True)
    for f in ["train.csv", "train_series.csv", "test.csv", "test_series.csv", "sample_submission.csv"]:
        r = subprocess.run(["kaggle", "competitions", "download", "-c", COMPETITION,
                           "-f", f], capture_output=True, text=True)
        src = pathlib.Path(f)
        if r.returncode == 0 and src.exists():
            src.rename(dst / f)
            print("downloaded", f)
        else:
            print("skip", f, r.stderr.strip()[:120])
    cfg = yaml.safe_load(pathlib.Path("config.yaml").read_text(encoding="utf-8"))
    cfg["data_dir"] = str(dst)
    pathlib.Path("config.yaml").write_text(yaml.safe_dump(cfg), encoding="utf-8")
    print("data_dir set to", dst)

In [ ]:
!mkdir -p outputs/models
!python train.py --config config.yaml --folds $FOLDS_TO_TRAIN --use_wandb $USE_WANDB --create_oof $CREATE_OOF

In [ ]:
# 训练产物打包上传到 Kaggle Dataset: {user}/exp-results-{exp}-{child}
import json, os, re, shutil, subprocess

def kaggle_user():
    u = os.environ.get("KAGGLE_USERNAME")
    if u:
        return u
    r = subprocess.run(["kaggle", "config", "view"], capture_output=True, text=True)
    m = re.search(r"username:\s*(\S+)", r.stdout + r.stderr)
    return m.group(1) if m else None

user = kaggle_user()
if user is None:
    raise SystemExit("cannot determine KAGGLE_USERNAME in kernel")

src = "/kaggle/working/outputs"
pkg = "/kaggle/working/dataset_pkg"
shutil.rmtree(pkg, ignore_errors=True)
os.makedirs(pkg, exist_ok=True)
for f in os.listdir(src):
    s = os.path.join(src, f)
    if os.path.isdir(s):
        shutil.copytree(s, os.path.join(pkg, f))
    else:
        shutil.copy(s, os.path.join(pkg, f))

slug = f"exp-results-{EXP_NAME.lower()}-{CHILD_EXP_NAME}"
meta = {
    "id": f"{user}/{slug}",
    "title": f"exp-results {EXP_NAME} {CHILD_EXP_NAME}",
    "licenses": [{"name": "other"}],
}
with open(os.path.join(pkg, "dataset-metadata.json"), "w") as fh:
    json.dump(meta, fh)

r = subprocess.run(["kaggle", "datasets", "create", "-p", pkg,
                   "--dir-mode", "tar", "-q"],
                   capture_output=True, text=True)
if r.returncode != 0 and "already exists" not in (r.stdout + r.stderr):
    print("create output:", r.stdout, r.stderr)
r = subprocess.run(["kaggle", "datasets", "version", "-p", pkg,
                   "-m", f"{EXP_NAME} {CHILD_EXP_NAME}", "--dir-mode", "tar", "-q"],
                   capture_output=True, text=True)
print("version output:", r.stdout, r.stderr)
print("UPLOAD_DONE", slug)